# 03 — Construcción de la tabla Gold para AndinaLog 03B

## Objetivo

Crear una tabla preparada para el EDA y el modelo predictivo. Cada fila continúa representando **una lectura de telemetría**.

Se construirán tres variables temporales:

1. `temperatura_anterior`: temperatura de la lectura inmediatamente anterior del mismo viaje.
2. `variacion_temperatura_30min`: cambio de temperatura respecto de aproximadamente 30 minutos antes.
3. `desviacion_proximos_60min_flag`: indica si aparece una desviación durante los siguientes 60 minutos.

El futuro se utiliza únicamente para construir la variable objetivo; nunca se utiliza como predictor.

## 1. Librerías, rutas y reglas

In [1]:
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)

CONFIG = {
    "ruta_silver": Path("outputs/02_bronze_silver_iot/andinalog_iot_telemetry_silver.csv"),
    "carpeta_salida": Path("outputs/03_silver_gold_iot"),
    "entidad": "viaje_id",
    "timestamp": "timestamp",
    "ventana_variacion_min": 30,
    "horizonte_objetivo_min": 60,
    "tolerancia_min": 5,
    "predictores_minimos": [
        "temperatura_cabina_c",
        "humedad_cabina_pct",
        "desviacion_termica_flag",
        "temperatura_anterior",
        "variacion_temperatura_30min",
    ],
}

CONFIG["carpeta_salida"].mkdir(parents=True, exist_ok=True)
RUTA_GOLD = CONFIG["carpeta_salida"] / "andinalog_iot_modelado_gold.csv"
RUTA_INFORME = CONFIG["carpeta_salida"] / "Informe_03_Silver_Gold_IoT.md"

print("Ventana de variación:", CONFIG["ventana_variacion_min"], "minutos")
print("Horizonte del objetivo:", CONFIG["horizonte_objetivo_min"], "minutos")
print("Tolerancia temporal:", CONFIG["tolerancia_min"], "minutos")

Ventana de variación: 30 minutos
Horizonte del objetivo: 60 minutos
Tolerancia temporal: 5 minutos


### Explicación sencilla

Como las lecturas normalmente aparecen cada 30 minutos, se acepta una diferencia máxima de 5 minutos respecto del momento esperado. Esta tolerancia evita relacionar lecturas demasiado alejadas en el tiempo.

## 2. Carga y revisión del Silver

In [2]:
ruta_silver = CONFIG["ruta_silver"]
if not ruta_silver.exists():
    alternativa = Path("/content/andinalog_iot_telemetry_silver.csv")
    if alternativa.exists():
        ruta_silver = alternativa
    else:
        raise FileNotFoundError("No se encontró el CSV Silver. Ajusta CONFIG['ruta_silver'].")

silver = pd.read_csv(ruta_silver)
silver["timestamp"] = pd.to_datetime(silver["timestamp"], errors="coerce", utc=True)

columnas_necesarias = [
    "_fila_bronze", "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "humedad_cabina_pct", "desviacion_termica_flag",
    "desviacion_proximos_60min_flag",
]
faltantes = [c for c in columnas_necesarias if c not in silver.columns]
if faltantes:
    raise ValueError(f"Faltan columnas necesarias en Silver: {faltantes}")

print(f"Silver: {len(silver):,} filas y {silver.shape[1]} columnas")
print("Viajes:", silver["viaje_id"].nunique())
print("Timestamps no interpretables:", int(silver["timestamp"].isna().sum()))
print("Duplicados por viaje + timestamp:", int(silver.duplicated(["viaje_id", "timestamp"]).sum()))

Silver: 28,465 filas y 41 columnas
Viajes: 1200
Timestamps no interpretables: 0
Duplicados por viaje + timestamp: 0


## 3. Frecuencia observada de las lecturas

In [3]:
ordenado = silver.sort_values(["viaje_id", "timestamp"]).copy()
intervalos = ordenado.groupby("viaje_id")["timestamp"].diff().dt.total_seconds().div(60).dropna()

print("Intervalos más frecuentes entre lecturas del mismo viaje:")
display(intervalos.value_counts().head(10).rename_axis("minutos").to_frame("cantidad"))
print("Intervalo mediano:", float(intervalos.median()), "minutos")

Intervalos más frecuentes entre lecturas del mismo viaje:


,cantidad
minutos,
30.0,26961
60.0,302
90.0,2


Intervalo mediano: 30.0 minutos


### Interpretación sencilla

La mayoría de las lecturas está separada por 30 minutos. Por eso tiene sentido buscar una lectura cercana a 30 minutos atrás para calcular la tendencia y exigir información cercana a 60 minutos después para completar la ventana del objetivo.

## 4. Función para crear las variables temporales

In [4]:
def construir_variables_temporales(df, config):
    '''Construye rezagos y objetivo dentro de cada viaje, sin mezclar entidades.'''
    salida = df.sort_values([config["entidad"], config["timestamp"]]).reset_index(drop=True).copy()

    n = len(salida)
    temperatura_anterior = np.full(n, np.nan)
    ts_temperatura_anterior = np.full(n, np.datetime64("NaT"), dtype="datetime64[ns]")
    variacion_30 = np.full(n, np.nan)
    ts_base_variacion = np.full(n, np.datetime64("NaT"), dtype="datetime64[ns]")
    ventana_evaluable = np.zeros(n, dtype=bool)
    objetivo = np.full(n, np.nan)

    delta_30 = pd.Timedelta(minutes=config["ventana_variacion_min"])
    delta_60 = pd.Timedelta(minutes=config["horizonte_objetivo_min"])
    tolerancia = pd.Timedelta(minutes=config["tolerancia_min"])

    for _, indices in salida.groupby(config["entidad"], sort=False).groups.items():
        indices = np.asarray(list(indices), dtype=int)
        grupo = salida.loc[indices]
        tiempos = pd.DatetimeIndex(grupo[config["timestamp"]])
        temperaturas = grupo["temperatura_cabina_c"].to_numpy(dtype=float)
        flags = grupo["desviacion_termica_flag"].to_numpy(dtype=int)

        # Lectura inmediatamente anterior del mismo viaje.
        if len(indices) > 1:
            temperatura_anterior[indices[1:]] = temperaturas[:-1]
            ts_temperatura_anterior[indices[1:]] = tiempos[:-1].tz_localize(None).to_numpy()

        # Lectura más cercana a t - 30 minutos, dentro de la tolerancia.
        posiciones_30 = tiempos.get_indexer(tiempos - delta_30, method="nearest", tolerance=tolerancia)
        tiene_base = posiciones_30 >= 0
        filas_30 = indices[tiene_base]
        bases_30 = posiciones_30[tiene_base]
        variacion_30[filas_30] = temperaturas[tiene_base] - temperaturas[bases_30]
        ts_base_variacion[filas_30] = tiempos[bases_30].tz_localize(None).to_numpy()

        # La ventana es evaluable si existe una lectura cercana a t + 60 minutos.
        posiciones_60 = tiempos.get_indexer(tiempos + delta_60, method="nearest", tolerance=tolerancia)
        tiene_cobertura = posiciones_60 >= 0
        ventana_evaluable[indices] = tiene_cobertura

        # Objetivo: máximo flag actual observado dentro de (t, t + 60 min].
        for posicion in np.flatnonzero(tiene_cobertura):
            en_ventana = (tiempos > tiempos[posicion]) & (tiempos <= tiempos[posicion] + delta_60)
            objetivo[indices[posicion]] = int(flags[en_ventana].max(initial=0))

    salida["temperatura_anterior"] = temperatura_anterior
    salida["ts_temperatura_anterior"] = pd.to_datetime(ts_temperatura_anterior, utc=True)
    salida["variacion_temperatura_30min"] = variacion_30
    salida["ts_base_variacion"] = pd.to_datetime(ts_base_variacion, utc=True)
    salida["ventana_objetivo_evaluable"] = ventana_evaluable
    salida["desviacion_proximos_60min_flag_calculada"] = pd.array(objetivo, dtype="Int64")
    return salida

## 5. Construcción de Gold

In [5]:
trabajo = construir_variables_temporales(silver, CONFIG)

# Se conserva la etiqueta recibida solo para comprobarla; la etiqueta Gold es la calculada.
trabajo = trabajo.rename(columns={
    "desviacion_proximos_60min_flag": "objetivo_fuente_flag"
})
trabajo["desviacion_proximos_60min_flag"] = trabajo[
    "desviacion_proximos_60min_flag_calculada"
]
trabajo["objetivo_coincide_con_fuente"] = (
    trabajo["desviacion_proximos_60min_flag"].isna()
    | trabajo["desviacion_proximos_60min_flag"].eq(trabajo["objetivo_fuente_flag"])
)

columnas_gold = [
    "_fila_bronze", "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "humedad_cabina_pct", "desviacion_termica_flag",
    "temperatura_anterior", "ts_temperatura_anterior",
    "variacion_temperatura_30min", "ts_base_variacion",
    "ventana_objetivo_evaluable", "desviacion_proximos_60min_flag",
    "objetivo_fuente_flag", "objetivo_coincide_con_fuente",
]
gold = trabajo[columnas_gold].copy()

print(f"Gold construido: {len(gold):,} filas y {gold.shape[1]} columnas")
display(gold.head())

Gold construido: 28,465 filas y 17 columnas


,_fila_bronze,timestamp,viaje_id,order_id,camion_id,producto_id,temperatura_cabina_c,humedad_cabina_pct,desviacion_termica_flag,temperatura_anterior,ts_temperatura_anterior,variacion_temperatura_30min,ts_base_variacion,ventana_objetivo_evaluable,desviacion_proximos_60min_flag,objetivo_fuente_flag,objetivo_coincide_con_fuente
0,1,2026-08-11 18:11:00+00:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.19,72.6,0,NaN,NaT,NaN,NaT,True,0,0,True
1,2,2026-08-11 18:41:00+00:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,4.91,75.0,0,2.19,2026-08-11 18:11:00+00:00,2.72,2026-08-11 18:11:00+00:00,True,0,0,True
2,3,2026-08-11 19:11:00+00:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.76,94.7,0,4.91,2026-08-11 18:41:00+00:00,-2.15,2026-08-11 18:41:00+00:00,True,0,0,True
3,4,2026-08-11 19:41:00+00:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,3.96,74.0,0,2.76,2026-08-11 19:11:00+00:00,1.20,2026-08-11 19:11:00+00:00,True,0,0,True
4,5,2026-08-11 20:11:00+00:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,4.49,70.3,0,3.96,2026-08-11 19:41:00+00:00,0.53,2026-08-11 19:41:00+00:00,True,0,0,True


### Explicación sencilla

La tabla conserva una fila por lectura. La columna `objetivo_fuente_flag` queda únicamente como comprobación; el análisis posterior utilizará el objetivo reconstruido a partir de las desviaciones observadas dentro del mismo viaje.

## 6. Cobertura de las variables nuevas

In [6]:
resumen_cobertura = pd.DataFrame({
    "variable": [
        "temperatura_anterior", "variacion_temperatura_30min",
        "ventana_objetivo_evaluable", "desviacion_proximos_60min_flag",
    ],
    "valores_disponibles": [
        int(gold["temperatura_anterior"].notna().sum()),
        int(gold["variacion_temperatura_30min"].notna().sum()),
        int(gold["ventana_objetivo_evaluable"].sum()),
        int(gold["desviacion_proximos_60min_flag"].notna().sum()),
    ],
})
resumen_cobertura["porcentaje"] = (
    resumen_cobertura["valores_disponibles"] / len(gold) * 100
).round(2)
display(resumen_cobertura)

print("Objetivo calculado:")
display(
    gold["desviacion_proximos_60min_flag"]
    .value_counts(dropna=False)
    .rename_axis("clase")
    .to_frame("lecturas")
)

,variable,valores_disponibles,porcentaje
0,temperatura_anterior,27265,95.78
1,variacion_temperatura_30min,26961,94.72
2,ventana_objetivo_evaluable,25797,90.63
3,desviacion_proximos_60min_flag,25797,90.63


Objetivo calculado:


,lecturas
clase,
0,24501
<NA>,2668
1,1296


In [7]:
print("Interpretación sencilla")
print("- La primera lectura de cada viaje no tiene una temperatura anterior.")
print("- Si falta una lectura cercana a 30 minutos atrás, la variación queda vacía.")
print("- Las últimas lecturas de cada viaje no tienen 60 minutos futuros completos.")
print("- En esas ventanas incompletas el objetivo queda vacío, no se asigna automáticamente un cero.")

Interpretación sencilla
- La primera lectura de cada viaje no tiene una temperatura anterior.
- Si falta una lectura cercana a 30 minutos atrás, la variación queda vacía.
- Las últimas lecturas de cada viaje no tienen 60 minutos futuros completos.
- En esas ventanas incompletas el objetivo queda vacío, no se asigna automáticamente un cero.


## 7. Comparación con la etiqueta recibida

In [8]:
comparables = gold["desviacion_proximos_60min_flag"].notna()
coinciden = gold.loc[comparables, "objetivo_coincide_con_fuente"]

print("Ventanas comparables:", int(comparables.sum()))
print("Coincidencias:", int(coinciden.sum()))
print("Diferencias:", int((~coinciden).sum()))

if (~coinciden).any():
    print("\nLecturas con diferencia:")
    display(gold.loc[comparables & ~gold["objetivo_coincide_con_fuente"], [
        "_fila_bronze", "viaje_id", "timestamp", "objetivo_fuente_flag",
        "desviacion_proximos_60min_flag",
    ]])

Ventanas comparables: 25797
Coincidencias: 25793
Diferencias: 4

Lecturas con diferencia:


,_fila_bronze,viaje_id,timestamp,objetivo_fuente_flag,desviacion_proximos_60min_flag
451,458,VIA-00020,2026-08-26 00:54:00+00:00,1,0
17322,17533,VIA-00731,2026-08-04 01:25:00+00:00,1,0
24734,25028,VIA-01043,2026-08-06 05:00:00+00:00,1,0
25330,25629,VIA-01068,2026-08-08 01:58:00+00:00,1,0


### Interpretación sencilla

La comparación permite verificar la etiqueta sin copiarla ciegamente. Una diferencia puede aparecer porque Gold se construye exclusivamente con las lecturas aprobadas en Silver; una lectura que quedó en cuarentena ya no puede aportar un evento futuro.

## 8. Controles finales

In [9]:
predictores = CONFIG["predictores_minimos"]
columnas_futuras = {
    "desviacion_proximos_60min_flag", "objetivo_fuente_flag",
    "ventana_objetivo_evaluable", "objetivo_coincide_con_fuente",
}

orden_correcto = bool(
    gold.groupby("viaje_id")["timestamp"]
    .apply(lambda s: s.is_monotonic_increasing)
    .all()
)
antecedentes_validos = bool(
    (gold["ts_temperatura_anterior"].dropna() < gold.loc[
        gold["ts_temperatura_anterior"].notna(), "timestamp"
    ]).all()
)
bases_validas = bool(
    (gold["ts_base_variacion"].dropna() < gold.loc[
        gold["ts_base_variacion"].notna(), "timestamp"
    ]).all()
)

controles = pd.DataFrame([
    {"control": "Una fila Gold por fila Silver", "resultado": len(gold) == len(silver),
     "detalle": f"{len(gold)} = {len(silver)}"},
    {"control": "Clave viaje + timestamp única", "resultado": not gold.duplicated(["viaje_id", "timestamp"]).any(),
     "detalle": f"duplicados: {int(gold.duplicated(['viaje_id', 'timestamp']).sum())}"},
    {"control": "Orden temporal correcto por viaje", "resultado": orden_correcto, "detalle": "orden ascendente"},
    {"control": "Temperatura anterior pertenece al pasado", "resultado": antecedentes_validos, "detalle": "timestamp anterior < timestamp actual"},
    {"control": "Base de variación pertenece al pasado", "resultado": bases_validas, "detalle": "base temporal < timestamp actual"},
    {"control": "Objetivo nulo cuando la ventana no es evaluable",
     "resultado": gold.loc[~gold["ventana_objetivo_evaluable"], "desviacion_proximos_60min_flag"].isna().all(),
     "detalle": f"ventanas no evaluables: {int((~gold['ventana_objetivo_evaluable']).sum())}"},
    {"control": "Predictores mínimos disponibles", "resultado": set(predictores).issubset(gold.columns),
     "detalle": ", ".join(predictores)},
    {"control": "Sin variables futuras entre predictores", "resultado": not bool(set(predictores) & columnas_futuras),
     "detalle": "el objetivo no es predictor"},
])
display(controles)

if not controles["resultado"].all():
    raise AssertionError("Uno o más controles Gold no se cumplieron.")
print("Todos los controles Gold se cumplieron.")

,control,resultado,detalle
0,Una fila Gold por fila Silver,True,28465 = 28465
1,Clave viaje + timestamp única,True,duplicados: 0
2,Orden temporal correcto por viaje,True,orden ascendente
3,Temperatura anterior pertenece al pasado,True,timestamp anterior < timestamp actual
4,Base de variación pertenece al pasado,True,base temporal < timestamp actual
5,Objetivo nulo cuando la ventana no es evaluable,True,ventanas no evaluables: 2668
6,Predictores mínimos disponibles,True,"temperatura_cabina_c, humedad_cabina_pct, desv..."
7,Sin variables futuras entre predictores,True,el objetivo no es predictor


Todos los controles Gold se cumplieron.


## 9. Exportación de Gold e informe

In [10]:
for columna in ["timestamp", "ts_temperatura_anterior", "ts_base_variacion"]:
    gold[columna] = gold[columna].astype("string")

gold.to_csv(RUTA_GOLD, index=False, encoding="utf-8-sig")

conteo_objetivo = gold["desviacion_proximos_60min_flag"].value_counts(dropna=False)
informe = f'''# Informe 03 — Construcción Silver a Gold IoT

## Objetivo

Construir una tabla predictiva por lectura sin mezclar viajes ni utilizar datos futuros como predictores.

## Resultados

- Filas Silver: {len(silver):,}
- Filas Gold: {len(gold):,}
- Viajes: {gold['viaje_id'].nunique():,}
- Temperaturas anteriores disponibles: {gold['temperatura_anterior'].notna().sum():,}
- Variaciones de 30 minutos disponibles: {gold['variacion_temperatura_30min'].notna().sum():,}
- Ventanas de 60 minutos evaluables: {gold['ventana_objetivo_evaluable'].sum():,}
- Ventanas no evaluables: {(~gold['ventana_objetivo_evaluable']).sum():,}
- Objetivos positivos: {(gold['desviacion_proximos_60min_flag'] == 1).sum():,}
- Objetivos negativos: {(gold['desviacion_proximos_60min_flag'] == 0).sum():,}
- Diferencias frente a la etiqueta recibida: {int((comparables & ~gold['objetivo_coincide_con_fuente']).sum()):,}

## Definiciones

- `temperatura_anterior`: lectura inmediatamente anterior del mismo viaje.
- `variacion_temperatura_30min`: temperatura actual menos la lectura cercana a 30 minutos atrás, con tolerancia de {CONFIG['tolerancia_min']} minutos.
- `desviacion_proximos_60min_flag`: vale 1 si existe una desviación en `(t, t+60 min]`; vale 0 si la ventana está completa y no aparece una desviación.
- `ventana_objetivo_evaluable`: confirma que existe una lectura cercana al final de los 60 minutos.

## Conciliación

Gold conserva exactamente una fila por cada fila Silver: {len(gold):,} = {len(silver):,}.

## Limitación

La reconstrucción utiliza únicamente lecturas Silver. Si una lectura con una desviación quedó en cuarentena, el evento no puede observarse desde Gold y puede diferir de la etiqueta recibida en Bronze.

Ejecución UTC: {datetime.now(timezone.utc).isoformat()}
'''
RUTA_INFORME.write_text(informe, encoding="utf-8")

print("Gold generado:", RUTA_GOLD.resolve())
print("Informe generado:", RUTA_INFORME.resolve())

Gold generado: C:\Users\remrodri\Documents\Codex\2026-09-28\referenced-chatgpt-conversation-this-is-an\outputs\03_silver_gold_iot\andinalog_iot_modelado_gold.csv
Informe generado: C:\Users\remrodri\Documents\Codex\2026-09-28\referenced-chatgpt-conversation-this-is-an\outputs\03_silver_gold_iot\Informe_03_Silver_Gold_IoT.md


## 10. Conclusión sencilla

La tabla Gold conserva todas las lecturas aprobadas en Silver y agrega solamente la información temporal necesaria para la tarea. Ya está preparada para el siguiente notebook: **EDA y gráficos obligatorios**.